# RAG-based Document Question Answering Chatbot
### A beginner-friendly, step-by-step project

**What we're building:** Upload any document (PDF or text), then ask questions about it in plain English — the chatbot reads the document and answers using it, instead of making things up.

**This technique is called RAG — Retrieval-Augmented Generation:**
1. **Retrieval** — find the most relevant parts of your document for a question
2. **Augmented Generation** — give those relevant parts to an AI model so it answers *using your document*, not just its general knowledge

**Steps in this notebook:**
1. Install libraries
2. Set up your free Gemini API key
3. Upload a document
4. Split it into chunks
5. Convert chunks into embeddings (numbers that capture meaning) and store in FAISS
6. Ask questions and get answers grounded in the document
7. Save everything for the Streamlit app

Run each cell **in order**, top to bottom (Shift + Enter).

## Step 1: Install libraries

In [ ]:
!pip install -q langchain langchain-community langchain-google-genai faiss-cpu pypdf tiktoken

## Step 2: Add your Gemini API key

Paste the API key you copied from Google AI Studio (aistudio.google.com) when this cell asks for it. It's hidden as you type — that's normal, it's just like a password field.

In [ ]:
import getpass
import os

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key here: ")
print("Key saved for this session.")

## Step 3: Upload a document

Any PDF or .txt file works — a notes file, an assignment, an article, your resume, anything. Run the cell and click 'Choose Files' to upload.

In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print(f"Uploaded: {file_name}")

## Step 4: Load the document
We read the file's text content so the model can work with it.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader

if file_name.lower().endswith(".pdf"):
    loader = PyPDFLoader(file_name)
else:
    loader = TextLoader(file_name)

documents = loader.load()
print(f"Loaded {len(documents)} page(s)/section(s).")
print("\nPreview of first 300 characters:\n")
print(documents[0].page_content[:300])

## Step 5: Split the document into chunks

Why? Documents can be long, and AI models can only "read" a limited amount of text at once. So we break the document into small overlapping chunks — each chunk is a self-contained piece the model can search through and read individually.

`chunk_size=1000` means each chunk is about 1000 characters. `chunk_overlap=200` means chunks share 200 characters with their neighbor, so we don't accidentally cut an important sentence in half between two chunks.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

print(f"Document split into {len(chunks)} chunks.")
print("\nExample chunk:\n")
print(chunks[0].page_content[:300])

## Step 6: Turn chunks into embeddings, store in FAISS

**Embeddings** = converting text into a list of numbers that captures its *meaning* (not just the words). Similar meanings end up as similar numbers.

**FAISS** = a fast search database for these number-lists (vectors). When you ask a question, FAISS finds the chunks whose meaning is closest to your question's meaning — even if the exact words are different.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

vector_store = FAISS.from_documents(chunks, embeddings)
print("Vector store created — your document is now searchable by meaning.")

## Step 7: Build the retrieval pipeline + connect the LLM

This sets up the full RAG chain:
1. Take your question
2. Find the most relevant chunks from FAISS (`k=3` means "find the top 3 most relevant chunks")
3. Give those chunks + your question to Gemini
4. Gemini answers using only that context — reducing made-up (hallucinated) answers

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.2)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

print("RAG pipeline ready. Ask questions in the next cell.")

## Step 8: Ask a question!
Change the question below to anything about your uploaded document.

In [ ]:
question = "What is this document about? Summarize it in 3 sentences."

result = qa_chain.invoke({"query": question})

print("Question:", question)
print("\nAnswer:\n", result["result"])

In [ ]:
# Try another question of your own
question = "Ask your own question here"

result = qa_chain.invoke({"query": question})
print("Answer:\n", result["result"])

## Step 9: See which parts of the document were used

This shows you exactly which chunks the model pulled from to answer — useful for checking the answer isn't made up.

In [ ]:
for i, doc in enumerate(result["source_documents"], 1):
    print(f"--- Source chunk {i} ---")
    print(doc.page_content[:300])
    print()

## Step 10: Save the vector store (for the Streamlit app)

This saves your FAISS index to a folder so the deployed app can load it instantly without re-processing the document every time.

In [ ]:
vector_store.save_local("faiss_index")
print("Saved to 'faiss_index' folder.")
print("Download this whole folder from the Colab file browser (left sidebar, folder icon):")
print("Right-click 'faiss_index' -> Download (it will download as a .zip).")

## Next step: Deployment

Download the `faiss_index` folder (as a zip) from Colab's file panel.

Then use the separate `app.py` file (provided alongside this notebook) to run an interactive Streamlit chat app where anyone can upload a document and ask questions in real time.

```bash
pip install streamlit langchain langchain-community langchain-google-genai faiss-cpu pypdf
streamlit run app.py
```

You'll need your same free Gemini API key when running the app.